In [1]:
# =====================================================================
# Improving linear models for used-car price prediction
# Works in Jupyter / VS Code (each "# %%" is a cell) or as a plain script.
# Needs scikit-learn >= 1.4 (TargetEncoder, RMSLE scorer), pandas, scipy.
# =====================================================================

# %% 0. Setup ---------------------------------------------------------
import re
import time
import warnings

import numpy as np
import pandas as pd
from pandas.api.types import is_numeric_dtype
from scipy import sparse

from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin, clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import (
    ElasticNet, HuberRegressor, Lasso, LinearRegression,
    QuantileRegressor, Ridge,
)
from sklearn.metrics import (
    mean_absolute_error, mean_absolute_percentage_error,
    mean_squared_error, mean_squared_log_error,
    median_absolute_error, r2_score,
)
from sklearn.model_selection import (
    GridSearchCV, KFold, cross_validate, train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder, SplineTransformer, StandardScaler, TargetEncoder,
)

warnings.filterwarnings("ignore")

# ---- EDIT THESE to match your column names (set to None if you have none)
CSV_PATH = "used_cars_cleaned_07.csv"
TARGET = "price_gbp"
YEAR_COL = "year"
MILEAGE_COL = "mileage"
ENGINE_COL = "engine_size"
MAKE_COL = "make"          # used for the make x age / make x mileage slopes
TEXT_COL = "version"       # free-text trim/version string
N_JOBS = 2
RANDOM_STATE = 42

# Words searched for inside the `version` text (edit freely)
VERSION_KEYWORDS = [
    "gt", "gti", "sport", "se", "sel", "auto", "manual", "turbo", "hybrid",
    "awd", "4x4", "amg", "m sport", "s line", "quattro", "tdi", "dsg",
    "premium", "luxury", "edition",
]

df = pd.read_csv(CSV_PATH)
X = df.drop(columns=[TARGET, "condition"], errors="ignore")
y = df[TARGET]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)  # same split as your notebook

def _present(col):
    return col if (col is not None and col in X.columns) else None

YEAR_COL, MILEAGE_COL, ENGINE_COL, MAKE_COL, TEXT_COL = map(
    _present, (YEAR_COL, MILEAGE_COL, ENGINE_COL, MAKE_COL, TEXT_COL)
)
print("Columns:", list(X.columns))
print("Using ->", dict(year=YEAR_COL, mileage=MILEAGE_COL, engine=ENGINE_COL,
                       make=MAKE_COL, text=TEXT_COL))

cv = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

_cat_raw = [c for c in X_train.columns
            if not is_numeric_dtype(X_train[c]) and c != TEXT_COL]
HIGH_CARD = [c for c in _cat_raw if X_train[c].nunique() > 30]
_seg = [c for c in _cat_raw
        if c not in HIGH_CARD and c != MAKE_COL and 2 <= X_train[c].nunique() <= 8]
SEG_COL = _seg[0] if _seg else None
print("High-cardinality cols (target-encoded):", HIGH_CARD)
print("Segment column for piecewise models:", SEG_COL)

Columns: ['make', 'model', 'version', 'mileage_miles', 'reg_year', 'fuel', 'transmission', 'power_hp', 'body_type', 'body_color', 'doors', 'seats', 'previous_owners', 'drivetrain', 'emission_class', 'annual_tax_gbp', 'seller_type', 'seller_city', 'seller_county', 'co2_gmile']
Using -> {'year': None, 'mileage': None, 'engine': None, 'make': 'make', 'text': 'version'}
High-cardinality cols (target-encoded): ['make', 'model', 'seller_city', 'seller_county']
Segment column for piecewise models: transmission


In [2]:
# %% 1. Find the rows that are hurting RMSE ----------------------------
base_diag = Pipeline([
    ("prep", ColumnTransformer([
        ("num", Pipeline([("i", SimpleImputer(strategy="median")),
                          ("s", StandardScaler())]),
         X_train.select_dtypes("number").columns.tolist()),
        ("cat", Pipeline([("i", SimpleImputer(strategy="most_frequent")),
                          ("o", OneHotEncoder(handle_unknown="ignore"))]),
         _cat_raw),
    ])),
    ("model", Ridge(alpha=1)),
]).fit(X_train, y_train)

raw_pred = base_diag.predict(X_val)
err = pd.Series(np.abs(y_val.values - raw_pred), index=y_val.index)
worst = err.sort_values(ascending=False).head(20).index
print("\nNegative predictions:", int((raw_pred < 0).sum()))
print("\n20 worst validation rows:")
print(X_val.loc[worst].assign(actual=y_val.loc[worst],
                              predicted=pd.Series(raw_pred, index=y_val.index).loc[worst],
                              abs_err=err.loc[worst]).round(1).to_string())
print("\nShare of total squared error from the worst 20 rows: "
      f"{(err.loc[worst] ** 2).sum() / (err ** 2).sum():.1%}")
print("\nNumeric feature tails (look for impossible values):")
print(X_train.select_dtypes("number")
      .describe(percentiles=[.001, .01, .99, .999]).T.round(2).to_string())


Negative predictions: 3546

20 worst validation rows:
                 make      model                                                                     version  mileage_miles  reg_year           fuel transmission  power_hp    body_type body_color  doors  seats  previous_owners        drivetrain emission_class  annual_tax_gbp seller_type    seller_city    seller_county  co2_gmile   actual  predicted    abs_err
84760         Ferrari        F40                                       2.9 Coupe 2dr Petrol Manual (478 bhp)           8500      1992         Petrol       Manual         0        Coupe    Unknown      2      2                4  Rear Wheel Drive        Unknown           445.0       Trade        Preston       Lancashire        620  9999999  2296142.4  7703856.6
198368        Ferrari        275                                                                       GTB/4          27029      1968         Petrol       Manual         0        Coupe        Red      2      2            

In [3]:
# %% 2-4, 8. Building blocks -------------------------------------------
class Clipper(BaseEstimator, TransformerMixin):
    """Fix 2: clip each numeric column to training percentiles."""
    def __init__(self, lo=0.005, hi=0.995):
        self.lo, self.hi = lo, hi
    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.l_ = np.nanquantile(X, self.lo, axis=0)
        self.h_ = np.nanquantile(X, self.hi, axis=0)
        return self
    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.l_, self.h_)


class FeatureAdder(BaseEstimator, TransformerMixin):
    """Fix 4 + 8: age, mileage/yr, log features, structured `version` flags."""
    def __init__(self, year=None, mileage=None, engine=None, text=None,
                 engineer=True, keywords=()):
        self.year, self.mileage, self.engine = year, mileage, engine
        self.text, self.engineer, self.keywords = text, engineer, keywords
    def fit(self, X, y=None):
        self.ref_year_ = X[self.year].max() if self.year else None
        return self
    def transform(self, X):
        X = X.copy()
        if self.text:
            X[self.text] = X[self.text].fillna("").astype(str)
        if not self.engineer:
            return X
        if self.year:
            age = (self.ref_year_ - X[self.year]).clip(lower=0)
            X["age"] = age
            X = X.drop(columns=[self.year])
            if self.mileage:
                X["mileage_per_year"] = X[self.mileage] / (age + 1)
        if self.mileage:
            X["log_mileage"] = np.log1p(X[self.mileage].clip(lower=0))
        if self.engine:
            X["log_engine"] = np.log(X[self.engine].where(X[self.engine] > 0))
        if self.text:
            v = X[self.text].str.lower()
            for k in self.keywords:
                pat = rf"(?<![a-z0-9]){re.escape(k)}(?![a-z0-9])"
                X["ver_" + k.replace(" ", "_")] = v.str.contains(pat).astype(int)
            X["ver_engine_l"] = v.str.extract(r"(\d\.\d)")[0].astype(float)
        return X


class GroupSlopes(BaseEstimator, TransformerMixin):
    """Fix 4: make x age and make x log-mileage interactions.
    First column = group, remaining columns = numeric slopes."""
    def __init__(self, min_frequency=30):
        self.min_frequency = min_frequency
    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.cols_ = list(X.columns)
        self.ohe_ = OneHotEncoder(handle_unknown="infrequent_if_exist",
                                  min_frequency=self.min_frequency,
                                  sparse_output=True)
        self.ohe_.fit(self._g(X))
        N = X[self.cols_[1:]].astype(float)
        self.mu_, self.sd_ = N.mean(), N.std().replace(0, 1)
        return self
    def _g(self, X):
        return X[[self.cols_[0]]].fillna("missing").astype(str)
    def transform(self, X):
        X = pd.DataFrame(X)[self.cols_]
        G = self.ohe_.transform(self._g(X)).tocsr()
        N = X[self.cols_[1:]].astype(float).fillna(self.mu_)
        Z = ((N - self.mu_) / self.sd_).clip(-4, 4).to_numpy()
        blocks = [sparse.csr_matrix(G.multiply(Z[:, [j]]))
                  for j in range(Z.shape[1])]
        return sparse.hstack(blocks, format="csr")


def build_preprocessor(fe=True, clip=True, splines=True, interactions=True,
                       target_enc=True, text=None):
    num_sel = lambda d: [c for c in d.columns if is_numeric_dtype(d[c])
                         and not str(c).endswith("_missing")]
    ind_sel = lambda d: [c for c in d.columns if str(c).endswith("_missing")]
    cat_all = lambda d: [c for c in d.columns
                         if not is_numeric_dtype(d[c]) and c != TEXT_COL]
    high_sel = lambda d: [c for c in cat_all(d) if c in HIGH_CARD]
    low_sel = (lambda d: [c for c in cat_all(d) if c not in HIGH_CARD]) \
        if target_enc else cat_all
    spline_sel = lambda d: [c for c in ("age", MILEAGE_COL, ENGINE_COL)
                            if c and c in d.columns]
    slope_sel = lambda d: ([MAKE_COL] + [c for c in ("age", "log_mileage")
                                         if c in d.columns]
                           if MAKE_COL and any(c in d.columns
                                               for c in ("age", "log_mileage"))
                           else [])

    steps = [("imp", SimpleImputer(strategy="median"))]
    if clip:
        steps.append(("clip", Clipper()))
    steps.append(("scale", StandardScaler()))

    parts = [
        ("num", Pipeline(steps), num_sel),
        ("ind", "passthrough", ind_sel),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                  min_frequency=10)),
        ]), low_sel),
    ]
    if target_enc and HIGH_CARD:
        parts.append(("te", Pipeline([
            ("imp", SimpleImputer(strategy="constant", fill_value="missing")),
            ("enc", TargetEncoder(target_type="continuous",
                                  random_state=RANDOM_STATE)),
            ("scale", StandardScaler()),
        ]), high_sel))
    if splines:   # Fix 3: smooth curves that flatten at the ends
        parts.append(("spline", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("clip", Clipper()),
            ("spl", SplineTransformer(n_knots=6, degree=3, knots="uniform",
                                      extrapolation="constant")),
        ]), spline_sel))
    if interactions and fe:
        parts.append(("slopes", GroupSlopes(), slope_sel))
    if text == "tfidf" and TEXT_COL:
        parts.append(("tfidf", TfidfVectorizer(
            sublinear_tf=True, ngram_range=(1, 2), min_df=3,
            max_features=15000, strip_accents="unicode"), TEXT_COL))

    return Pipeline([
        ("fe", FeatureAdder(YEAR_COL, MILEAGE_COL, ENGINE_COL, TEXT_COL,
                            engineer=fe, keywords=tuple(VERSION_KEYWORDS))),
        ("ct", ColumnTransformer(parts, sparse_threshold=0.3)),
    ])

In [4]:
# %% 5, 7. Wrappers: loss / target / segments ---------------------------
LOG_CAP = float(np.log1p(y_train.max() * 1.2))

def safe_expm1(z):
    """expm1 with a cap, so log-model errors cannot explode into the millions."""
    return np.expm1(np.clip(z, 0, LOG_CAP))

def logt(estimator):
    """Fix 5: train on log(1 + price)."""
    return TransformedTargetRegressor(regressor=estimator, func=np.log1p,
                                      inverse_func=safe_expm1)

class ClippedRegressor(BaseEstimator, RegressorMixin):
    """Fix 1: never predict below 0 or above the highest training price."""
    def __init__(self, estimator):
        self.estimator = estimator
    def fit(self, X, y):
        self.est_ = clone(self.estimator).fit(X, y)
        self.hi_ = float(np.max(y))
        return self
    def predict(self, X):
        return np.clip(self.est_.predict(X), 0, self.hi_)

class RelativeErrorRegressor(BaseEstimator, RegressorMixin):
    """Fix 5: weighted least squares, weight = 1/price (cheap cars matter).
    Wrapped estimator must be a Pipeline whose last step is named 'model'."""
    def __init__(self, estimator, floor=500):
        self.estimator, self.floor = estimator, floor
    def fit(self, X, y):
        w = 1.0 / np.maximum(np.asarray(y, float), self.floor)
        w = w / w.mean()   # keep the Ridge penalty on the same scale
        self.est_ = clone(self.estimator).fit(X, y, model__sample_weight=w)
        return self
    def predict(self, X):
        return self.est_.predict(X)

class SegmentedRegressor(BaseEstimator, RegressorMixin):
    """Fix 7: one linear model per segment (+ a global fallback)."""
    def __init__(self, estimator, segment_col=None, min_rows=3000):
        self.estimator, self.segment_col, self.min_rows = (
            estimator, segment_col, min_rows)
    def fit(self, X, y):
        y = pd.Series(np.asarray(y), index=X.index)
        self.global_ = clone(self.estimator).fit(X, y)
        self.seg_ = {}
        if self.segment_col:
            for s, idx in X.groupby(self.segment_col).groups.items():
                if len(idx) >= self.min_rows:
                    self.seg_[s] = clone(self.estimator).fit(
                        X.loc[idx], y.loc[idx])
        return self
    def predict(self, X):
        pred = np.asarray(self.global_.predict(X), dtype=float)
        for s, m in self.seg_.items():
            mask = (X[self.segment_col] == s).to_numpy()
            if mask.any():
                pred[mask] = m.predict(X[mask])
        return pred

class Subsampled(BaseEstimator, RegressorMixin):
    """Train a slow estimator (e.g. QuantileRegressor) on a random subset."""
    def __init__(self, estimator, n=30000):
        self.estimator, self.n = estimator, n
    def fit(self, X, y):
        y = pd.Series(np.asarray(y), index=X.index)
        if len(X) > self.n:
            idx = X.sample(self.n, random_state=RANDOM_STATE).index
            X, y = X.loc[idx], y.loc[idx]
        self.est_ = clone(self.estimator).fit(X, y)
        return self
    def predict(self, X):
        return self.est_.predict(X)


def pipe(model, **kw):
    return Pipeline([("prep", build_preprocessor(**kw)), ("model", model)])

def metrics(y_true, pred):
    pred = np.maximum(np.asarray(pred, float), 0)
    return {
        "MAE": mean_absolute_error(y_true, pred),
        "MedAE": median_absolute_error(y_true, pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, pred)),
        "RMSLE": np.sqrt(mean_squared_log_error(y_true, pred)),
        "MAPE%": 100 * mean_absolute_percentage_error(y_true, pred),
        "R2": r2_score(y_true, pred),
    }

In [5]:
# %% Experiments (ablation: add one fix at a time) ----------------------
OFF = dict(fe=False, clip=False, splines=False, interactions=False,
           target_enc=False)
ALL = dict(fe=True, clip=True, splines=True, interactions=True,
           target_enc=True)

experiments = {
    "A  OLS baseline":              pipe(LinearRegression(), **OFF),
    "B  +clip features (fix 2)":    pipe(Ridge(alpha=1), **{**OFF, "clip": True}),
    "C  +age/log/version (fix 4,8)": pipe(Ridge(alpha=1), **{**OFF, "clip": True, "fe": True}),
    "D  +splines (fix 3)":          pipe(Ridge(alpha=1), **{**OFF, "clip": True, "fe": True, "splines": True}),
    "E  +make x age slopes (fix 4)": pipe(Ridge(alpha=1), **{**OFF, "clip": True, "fe": True, "splines": True, "interactions": True}),
    "F  +target encoding (fix 4)":  pipe(Ridge(alpha=1), **ALL),
    "G  F + log target (fix 5)":    logt(pipe(Ridge(alpha=1), **ALL)),
    "H  G with Huber loss (fix 5)": logt(pipe(HuberRegressor(alpha=1e-3, max_iter=300), **ALL)),
    "I  F with 1/price weights (fix 5)": RelativeErrorRegressor(pipe(Ridge(alpha=1), **ALL)),
    "J  G with Lasso (fix 6)":      logt(pipe(Lasso(alpha=1e-4, max_iter=3000), **ALL)),
    "K  G with ElasticNet (fix 6)": logt(pipe(ElasticNet(alpha=1e-4, l1_ratio=0.5, max_iter=3000), **ALL)),
    "L  G per segment (fix 7)":     SegmentedRegressor(logt(pipe(Ridge(alpha=1), **ALL)), segment_col=SEG_COL),
    "M  G + TF-IDF version (fix 8)": logt(pipe(Ridge(alpha=3), text="tfidf", **ALL)),
    "N  median regression, 30k rows (fix 5)": Subsampled(
        logt(pipe(QuantileRegressor(quantile=0.5, alpha=1e-4, solver="highs"), **ALL))),
}

# SPEED: models are fitted on a random sample of the training rows
# (validation is always the full validation set). Set to None for all rows.
FAST_N = 60000
if FAST_N and len(X_train) > FAST_N:
    X_fit = X_train.sample(FAST_N, random_state=RANDOM_STATE)
    y_fit = y_train.loc[X_fit.index]
else:
    X_fit, y_fit = X_train, y_train

rows, fitted = [], {}
for name, est in experiments.items():
    t0 = time.time()
    try:
        model = ClippedRegressor(est).fit(X_fit, y_fit)
        fitted[name] = model
        rows.append({"model": name,
                     **metrics(y_val, model.predict(X_val)),
                     "sec": time.time() - t0})
        print(f"done: {name}  ({time.time() - t0:.0f}s)")
    except Exception as e:
        print(f"FAILED: {name} -> {type(e).__name__}: {str(e)[:150]}")

holdout = pd.DataFrame(rows).set_index("model").round(3)
print("\nValidation results (sorted by MAE):")
print(holdout.sort_values("MAE").to_string())

done: A  OLS baseline  (3s)
done: B  +clip features (fix 2)  (2s)
done: C  +age/log/version (fix 4,8)  (7s)
done: D  +splines (fix 3)  (8s)
done: E  +make x age slopes (fix 4)  (7s)
done: F  +target encoding (fix 4)  (5s)
done: G  F + log target (fix 5)  (5s)
done: H  G with Huber loss (fix 5)  (21s)
done: I  F with 1/price weights (fix 5)  (5s)
done: J  G with Lasso (fix 6)  (16s)
done: K  G with ElasticNet (fix 6)  (44s)
done: L  G per segment (fix 7)  (12s)
done: M  G + TF-IDF version (fix 8)  (8s)
done: N  median regression, 30k rows (fix 5)  (146s)

Validation results (sorted by MAE):
                                             MAE     MedAE       RMSE  RMSLE   MAPE%     R2      sec
model                                                                                               
M  G + TF-IDF version (fix 8)           2546.625  1028.320  41823.973  0.194  12.824  0.232    7.854
H  G with Huber loss (fix 5)            3392.884  1406.880  42429.274  0.246  17.005  0.210   20.872

In [6]:
# %% 6. Tune the strongest families (wider, log-scaled grids) -----------
tuned = {}

def tune(name, est, grid):
    s = GridSearchCV(est, grid, scoring="neg_mean_absolute_error", cv=cv,
                     n_jobs=N_JOBS, pre_dispatch=N_JOBS, refit=True)
    s.fit(X_fit, y_fit)
    print(f"{name}: best {s.best_params_}  CV MAE £{-s.best_score_:,.0f}")
    tuned[name] = s.best_estimator_
    return s

tune("Ridge+log tuned",
     logt(pipe(Ridge(), **ALL)),
     {"regressor__model__alpha": np.logspace(-3, 2, 6)})
tune("Lasso+log tuned",
     logt(pipe(Lasso(max_iter=3000), **ALL)),
     {"regressor__model__alpha": [1e-5, 1e-4, 1e-3, 1e-2]})
tune("ElasticNet+log tuned",
     logt(pipe(ElasticNet(max_iter=3000), **ALL)),
     {"regressor__model__alpha": [1e-5, 1e-4, 1e-3],
      "regressor__model__l1_ratio": [0.2, 0.5, 0.8]})

for name, est in tuned.items():
    model = ClippedRegressor(est).fit(X_fit, y_fit)
    fitted[name] = model
    holdout.loc[name] = pd.Series({**metrics(y_val, model.predict(X_val)),
                                   "sec": np.nan}).round(3)

Ridge+log tuned: best {'regressor__model__alpha': np.float64(10.0)}  CV MAE £3,453
Lasso+log tuned: best {'regressor__model__alpha': 0.0001}  CV MAE £3,452
ElasticNet+log tuned: best {'regressor__model__alpha': 0.0001, 'regressor__model__l1_ratio': 0.8}  CV MAE £3,452


In [7]:
# %% 9. Fair comparison: 5-fold CV with several metrics ------------------
FINALISTS = list(holdout.sort_values("MAE").index[:4]) + ["A  OLS baseline"]
FINALISTS = list(dict.fromkeys(FINALISTS))

scoring = {
    "MAE": "neg_mean_absolute_error",
    "MedAE": "neg_median_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "RMSLE": "neg_root_mean_squared_log_error",
    "MAPE": "neg_mean_absolute_percentage_error",
    "R2": "r2",
}
cv5 = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for name in FINALISTS:
    est = tuned.get(name, experiments.get(name))
    res = cross_validate(ClippedRegressor(est), X_fit, y_fit, cv=cv5,
                         scoring=scoring, n_jobs=N_JOBS)
    cv_rows.append({"model": name, **{
        k: (-1 if k not in ("R2",) else 1) * res[f"test_{k}"].mean()
        for k in scoring}})
cv_table = pd.DataFrame(cv_rows).set_index("model").round(3)
cv_table["MAPE"] = (cv_table["MAPE"] * 100).round(2)
print("\n5-fold CV on training data (lower is better except R2; MAPE in %):")
print(cv_table.sort_values("MAE").to_string())


5-fold CV on training data (lower is better except R2; MAPE in %):
                                             MAE     MedAE       RMSE  RMSLE  MAPE     R2
model                                                                                    
M  G + TF-IDF version (fix 8)           2563.659  1041.913  22812.099  0.198  12.9  0.571
L  G per segment (fix 7)                3359.985  1452.206  23537.135  0.241  17.0  0.543
H  G with Huber loss (fix 5)            3365.353  1412.264  23784.584  0.248  16.9  0.533
N  median regression, 30k rows (fix 5)  3405.627  1408.593  24089.795  0.251  17.0  0.521
A  OLS baseline                         4996.871  2597.558  24228.739  2.076  40.6  0.515


In [8]:
# %% Best model: predict on new data ------------------------------------
best_name = holdout.sort_values("MAE").index[0]
best_model = fitted[best_name]
print("\nBest on validation MAE:", best_name)
# new_predictions = best_model.predict(new_df)   # new_df has the same columns as X


Best on validation MAE: M  G + TF-IDF version (fix 8)
